In [1]:
# ============================================================
# Hospital Readmission Cost Tracker — Google Colab version
# Paste this whole block into one Colab cell and click ▶ (Run).
# Pulls live Medicare (CMS) data, cleans it with SQL, checks
# data quality, and gives you the Tableau files as a zip.
# ============================================================
import json, logging, re, sqlite3, time, shutil, os
from datetime import datetime, timezone
import pandas as pd, requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s", force=True)
log = logging.getLogger("pipeline")

# ---------- Settings & business assumptions ----------
CMS_API = "https://data.cms.gov/provider-data/api/1/datastore/query"
DATASETS = {"readmissions": "9n3s-kdb3", "hospitals": "xubh-q36u"}
COST_PER_READMISSION = 15200   # AHRQ HCUP Statistical Brief #248 average readmission cost
SCENARIOS = [0.25, 0.50, 1.00] # share of the excess-readmission gap closed
OUT = "tableau_files"
ALIASES = {"citytown": "city_town", "countyparish": "county_parish",
           "provider_id": "facility_id", "hospital_name": "facility_name"}
REQUIRED = {
    "readmissions": ["facility_id", "facility_name", "state", "measure_name",
                     "number_of_discharges", "excess_readmission_ratio",
                     "predicted_readmission_rate", "expected_readmission_rate",
                     "number_of_readmissions", "start_date", "end_date"],
    "hospitals": ["facility_id", "facility_name", "city_town", "state",
                  "hospital_type", "hospital_ownership", "hospital_overall_rating"],
}

# ---------- 1. EXTRACT (Python) ----------
def normalize_columns(df):
    df.columns = [ALIASES.get(n, n) for n in
                  (re.sub(r"[^0-9a-zA-Z]+", "_", str(c)).strip("_").lower() for c in df.columns)]
    return df

def fetch_dataset(dataset_id, page=500):
    rows, offset, total = [], 0, None
    while True:
        for attempt in range(1, 5):
            try:
                r = requests.get(f"{CMS_API}/{dataset_id}/0",
                                 params={"limit": page, "offset": offset}, timeout=60)
                r.raise_for_status(); payload = r.json(); break
            except Exception as e:
                if attempt == 4: raise RuntimeError(f"CMS API failed: {e}")
                log.warning("Retrying after error: %s", e); time.sleep(2 ** attempt)
        batch = payload.get("results", []); total = payload.get("count", total)
        rows += batch
        log.info("  %s: fetched %s / %s rows", dataset_id, len(rows), total)
        if not batch or (total is not None and len(rows) >= int(total)): break
        offset += page
    return pd.DataFrame(rows)

def extract():
    frames = {}
    for name, ds in DATASETS.items():
        log.info("Downloading %s from CMS...", name)
        df = normalize_columns(fetch_dataset(ds)).astype(str)
        missing = [c for c in REQUIRED[name] if c not in df.columns]
        if missing:
            raise ValueError(f"{name}: missing {missing}. Columns found: {list(df.columns)}")
        frames[name] = df
    return frames

# ---------- 2. TRANSFORM (SQL models) ----------
SQL_MODELS = r"""
-- Staging: clean and type the raw CMS readmissions feed.
-- CMS uses text like 'N/A' and 'Too Few to Measure' inside numeric columns,
-- so every numeric field is parsed defensively: anything not starting with a
-- digit (or '.') becomes NULL instead of a misleading 0.
DROP TABLE IF EXISTS stg_readmissions;
CREATE TABLE stg_readmissions AS
SELECT
    TRIM(facility_id)                                        AS facility_id,
    TRIM(facility_name)                                      AS facility_name,
    UPPER(TRIM(state))                                       AS state,
    TRIM(measure_name)                                       AS measure_name,
    REPLACE(REPLACE(TRIM(measure_name), 'READM-30-', ''), '-HRRP', '') AS condition_code,
    CASE WHEN TRIM(number_of_discharges) GLOB '[0-9.]*'       THEN CAST(TRIM(number_of_discharges) AS REAL) END       AS discharges,
    CASE WHEN TRIM(number_of_readmissions) GLOB '[0-9.]*'     THEN CAST(TRIM(number_of_readmissions) AS REAL) END     AS readmissions,
    CASE WHEN TRIM(excess_readmission_ratio) GLOB '[0-9.]*'   THEN CAST(TRIM(excess_readmission_ratio) AS REAL) END   AS excess_readmission_ratio,
    CASE WHEN TRIM(predicted_readmission_rate) GLOB '[0-9.]*' THEN CAST(TRIM(predicted_readmission_rate) AS REAL) END AS predicted_rate_pct,
    CASE WHEN TRIM(expected_readmission_rate) GLOB '[0-9.]*'  THEN CAST(TRIM(expected_readmission_rate) AS REAL) END  AS expected_rate_pct,
    TRIM(start_date)                                         AS period_start,
    TRIM(end_date)                                           AS period_end
FROM raw_readmissions
WHERE TRIM(facility_id) <> '';

-- Staging: one clean row per hospital, used to enrich readmission metrics.
DROP TABLE IF EXISTS stg_hospitals;
CREATE TABLE stg_hospitals AS
SELECT
    TRIM(facility_id)          AS facility_id,
    TRIM(facility_name)        AS facility_name,
    TRIM(city_town)            AS city,
    UPPER(TRIM(state))         AS state,
    TRIM(hospital_type)        AS hospital_type,
    TRIM(hospital_ownership)   AS ownership,
    CASE WHEN TRIM(hospital_overall_rating) GLOB '[1-5]'
         THEN CAST(TRIM(hospital_overall_rating) AS INTEGER) END AS star_rating
FROM raw_hospitals
WHERE TRIM(facility_id) <> ''
GROUP BY TRIM(facility_id);   -- guard against duplicate hospital rows

-- Fact table: one row per hospital x condition with the business metrics.
--
-- Excess readmissions = discharges x (predicted rate - expected rate).
-- CMS's predicted rate is the hospital's own risk-adjusted rate; the expected
-- rate is what an average hospital would see with the same patients. Only the
-- positive gap counts as "excess". Cost uses the assumption in the params table.
DROP TABLE IF EXISTS fct_hospital_condition;
CREATE TABLE fct_hospital_condition AS
WITH p AS (SELECT value AS cost_per_readmission FROM params WHERE name = 'cost_per_readmission')
SELECT
    r.facility_id,
    COALESCE(h.facility_name, r.facility_name)          AS facility_name,
    COALESCE(h.state, r.state)                          AS state,
    h.city, h.hospital_type, h.ownership, h.star_rating,
    r.condition_code,
    CASE r.condition_code
        WHEN 'AMI'      THEN 'Heart attack'
        WHEN 'HF'       THEN 'Heart failure'
        WHEN 'PN'       THEN 'Pneumonia'
        WHEN 'COPD'     THEN 'COPD'
        WHEN 'CABG'     THEN 'Bypass surgery (CABG)'
        WHEN 'HIP-KNEE' THEN 'Hip/knee replacement'
        ELSE r.condition_code END                       AS condition_name,
    r.discharges,
    r.readmissions,
    r.excess_readmission_ratio,
    r.predicted_rate_pct,
    r.expected_rate_pct,
    r.period_start, r.period_end,
    CASE WHEN r.excess_readmission_ratio IS NULL THEN 'Not measured'
         WHEN r.excess_readmission_ratio > 1      THEN 'Worse than expected'
         ELSE 'At or better than expected' END          AS performance_band,
    CASE WHEN r.excess_readmission_ratio > 1 AND r.discharges IS NOT NULL
         THEN ROUND(r.discharges * (r.predicted_rate_pct - r.expected_rate_pct) / 100.0, 2)
         ELSE 0 END                                     AS excess_readmissions,
    CASE WHEN r.excess_readmission_ratio > 1 AND r.discharges IS NOT NULL
         THEN ROUND(r.discharges * (r.predicted_rate_pct - r.expected_rate_pct) / 100.0
                    * p.cost_per_readmission, 0)
         ELSE 0 END                                     AS est_excess_cost
FROM stg_readmissions r
CROSS JOIN p
LEFT JOIN stg_hospitals h ON h.facility_id = r.facility_id;

-- One row per hospital: totals, worst condition, and rankings.
DROP TABLE IF EXISTS hospital_summary;
CREATE TABLE hospital_summary AS
WITH totals AS (
    SELECT
        facility_id,
        MAX(facility_name) AS facility_name, MAX(state) AS state, MAX(city) AS city,
        MAX(hospital_type) AS hospital_type, MAX(ownership) AS ownership,
        MAX(star_rating)   AS star_rating,
        COUNT(excess_readmission_ratio)                              AS conditions_measured,
        SUM(CASE WHEN excess_readmission_ratio > 1 THEN 1 ELSE 0 END) AS conditions_worse_than_expected,
        SUM(COALESCE(discharges, 0))                                 AS total_discharges,
        ROUND(SUM(excess_readmissions), 1)                           AS excess_readmissions,
        SUM(est_excess_cost)                                         AS est_excess_cost,
        ROUND(AVG(excess_readmission_ratio), 4)                      AS avg_excess_ratio
    FROM fct_hospital_condition
    GROUP BY facility_id
),
worst AS (
    SELECT facility_id, condition_name AS worst_condition, excess_readmission_ratio AS worst_ratio,
           ROW_NUMBER() OVER (PARTITION BY facility_id ORDER BY excess_readmission_ratio DESC) AS rn
    FROM fct_hospital_condition
    WHERE excess_readmission_ratio IS NOT NULL
)
SELECT
    t.*,
    CASE WHEN t.conditions_worse_than_expected > 0 THEN 1 ELSE 0 END AS any_worse_than_expected,
    w.worst_condition,
    w.worst_ratio,
    RANK() OVER (PARTITION BY t.state ORDER BY t.est_excess_cost DESC) AS state_cost_rank,
    ROUND(PERCENT_RANK() OVER (ORDER BY t.avg_excess_ratio), 3)       AS national_ratio_percentile
FROM totals t
LEFT JOIN worst w ON w.facility_id = t.facility_id AND w.rn = 1;

-- One row per state for the dashboard map.
DROP TABLE IF EXISTS state_summary;
CREATE TABLE state_summary AS
SELECT
    state,
    COUNT(*)                                                   AS hospitals,
    SUM(any_worse_than_expected)                               AS hospitals_worse_than_expected,
    ROUND(100.0 * SUM(any_worse_than_expected) / COUNT(*), 1)  AS pct_hospitals_worse,
    ROUND(SUM(excess_readmissions), 0)                         AS excess_readmissions,
    SUM(est_excess_cost)                                       AS est_excess_cost,
    ROUND(AVG(avg_excess_ratio), 4)                            AS avg_excess_ratio,
    RANK() OVER (ORDER BY SUM(est_excess_cost) DESC)           AS national_cost_rank
FROM hospital_summary
WHERE conditions_measured > 0
GROUP BY state;

-- One row per condition: where the avoidable cost concentrates.
DROP TABLE IF EXISTS condition_summary;
CREATE TABLE condition_summary AS
SELECT
    condition_code,
    condition_name,
    COUNT(excess_readmission_ratio)                                 AS hospitals_measured,
    SUM(CASE WHEN excess_readmission_ratio > 1 THEN 1 ELSE 0 END)    AS hospitals_worse_than_expected,
    ROUND(100.0 * SUM(CASE WHEN excess_readmission_ratio > 1 THEN 1 ELSE 0 END)
          / NULLIF(COUNT(excess_readmission_ratio), 0), 1)          AS pct_worse,
    ROUND(SUM(excess_readmissions), 0)                              AS excess_readmissions,
    SUM(est_excess_cost)                                            AS est_excess_cost,
    ROUND(100.0 * SUM(est_excess_cost) / NULLIF(SUM(SUM(est_excess_cost)) OVER (), 0), 1) AS pct_of_total_cost
FROM fct_hospital_condition
GROUP BY condition_code, condition_name;

-- "What if" table for the executive story: savings if hospitals that are worse
-- than expected closed part of their gap.
DROP TABLE IF EXISTS savings_scenarios;
CREATE TABLE savings_scenarios AS
SELECT
    s.reduction_pct,
    CAST(ROUND(s.reduction_pct * 100) AS INTEGER) || '% of gap closed'  AS scenario,
    ROUND(s.reduction_pct * SUM(h.excess_readmissions), 0)              AS readmissions_avoided,
    ROUND(s.reduction_pct * SUM(h.est_excess_cost), 0)                  AS est_savings
FROM scenarios s
CROSS JOIN hospital_summary h
GROUP BY s.reduction_pct;

"""

def load_and_transform(conn, frames):
    for name, df in frames.items():
        df.to_sql(f"raw_{name}", conn, if_exists="replace", index=False)
    conn.executescript("""
        DROP TABLE IF EXISTS params; CREATE TABLE params (name TEXT PRIMARY KEY, value REAL);
        DROP TABLE IF EXISTS scenarios; CREATE TABLE scenarios (reduction_pct REAL);""")
    conn.execute("INSERT INTO params VALUES ('cost_per_readmission', ?)", (COST_PER_READMISSION,))
    conn.executemany("INSERT INTO scenarios VALUES (?)", [(s,) for s in SCENARIOS])
    conn.executescript(SQL_MODELS)
    conn.execute("""CREATE TABLE IF NOT EXISTS pipeline_runs
                    (run_at TEXT, readmission_rows INTEGER, hospital_rows INTEGER)""")
    conn.execute("INSERT INTO pipeline_runs VALUES (?,?,?)",
                 (datetime.now(timezone.utc).isoformat(timespec="seconds"),
                  len(frames["readmissions"]), len(frames["hospitals"])))
    conn.commit()

# ---------- 3. DATA QUALITY CHECKS ----------
def quality_checks(conn):
    q = lambda s: conn.execute(s).fetchone()[0]
    n = q("SELECT COUNT(*) FROM stg_readmissions")
    assert n > 0, "No readmission rows"
    assert q("SELECT COUNT(*) FROM stg_hospitals") > 0, "No hospital rows"
    dupes = q("""SELECT COUNT(*) FROM (SELECT facility_id, condition_code FROM stg_readmissions
                 GROUP BY 1,2 HAVING COUNT(*) > 1)""")
    assert dupes == 0, f"{dupes} duplicate hospital x condition rows"
    match = q("SELECT COUNT(*) FROM stg_readmissions JOIN stg_hospitals USING (facility_id)") / n
    assert match >= 0.90, f"Only {match:.0%} of rows matched a hospital"
    fact = q("SELECT SUM(est_excess_cost) FROM fct_hospital_condition")
    rollup = q("SELECT SUM(est_excess_cost) FROM hospital_summary")
    assert abs(fact - rollup) < 1, "Hospital rollup does not reconcile"
    log.info("Quality checks passed: %s rows, %.1f%% hospital match", n, 100 * match)

# ---------- 4. EXPORT for Tableau ----------
def export(conn):
    os.makedirs(OUT, exist_ok=True)
    for t in ["fct_hospital_condition", "hospital_summary", "state_summary",
              "condition_summary", "savings_scenarios", "pipeline_runs"]:
        pd.read_sql(f"SELECT * FROM {t}", conn).to_csv(f"{OUT}/{t}.csv", index=False)
    q = lambda s: conn.execute(s).fetchone()
    hosp, worse, excess, cost = q("""SELECT COUNT(*), SUM(any_worse_than_expected),
        SUM(excess_readmissions), SUM(est_excess_cost) FROM hospital_summary WHERE conditions_measured > 0""")
    headline = {
        "hospitals_measured": hosp,
        "hospitals_worse_than_expected": worse,
        "pct_worse": round(100 * worse / hosp, 1),
        "excess_readmissions": round(excess),
        "est_excess_cost": cost,
        "top_condition": q("SELECT condition_name, pct_of_total_cost FROM condition_summary ORDER BY est_excess_cost DESC LIMIT 1"),
        "top_state": q("SELECT state, est_excess_cost FROM state_summary ORDER BY est_excess_cost DESC LIMIT 1"),
        "savings_if_half_gap_closed": q("SELECT est_savings FROM savings_scenarios WHERE reduction_pct = 0.5")[0],
        "period": q("SELECT MIN(period_start), MAX(period_end) FROM stg_readmissions"),
        "conditions": [list(r) for r in conn.execute(
            "SELECT condition_name, est_excess_cost, pct_worse FROM condition_summary ORDER BY est_excess_cost DESC")],
        "top_states": [list(r) for r in conn.execute(
            "SELECT state, est_excess_cost, pct_hospitals_worse FROM state_summary ORDER BY est_excess_cost DESC LIMIT 5")],
    }
    json.dump(headline, open(f"{OUT}/headline_numbers.json", "w"), indent=2)
    shutil.make_archive("tableau_files", "zip", OUT)
    return headline

# ---------- RUN ----------
def run(frames=None):
    conn = sqlite3.connect("readmissions.db")
    frames = frames or extract()
    load_and_transform(conn, frames)
    quality_checks(conn)
    headline = export(conn)
    print("\n==== COPY EVERYTHING BELOW AND PASTE IT TO CLAUDE ====")
    print(json.dumps(headline, indent=2))
    print("==== END ====")
    return headline

if __name__ == "__main__" and "SKIP_RUN" not in globals():
    run()
    try:
        from google.colab import files
        files.download("tableau_files.zip")   # saves the Tableau files to your Downloads folder
    except ImportError:
        print("Tableau files saved to tableau_files.zip")


2026-09-30 00:19:34,039  Downloading readmissions from CMS...
2026-09-30 00:19:34,669    9n3s-kdb3: fetched 500 / 18330 rows
2026-09-30 00:19:36,442    9n3s-kdb3: fetched 1000 / 18330 rows
2026-09-30 00:19:37,004    9n3s-kdb3: fetched 1500 / 18330 rows
2026-09-30 00:19:37,605    9n3s-kdb3: fetched 2000 / 18330 rows
2026-09-30 00:19:38,174    9n3s-kdb3: fetched 2500 / 18330 rows
2026-09-30 00:19:38,738    9n3s-kdb3: fetched 3000 / 18330 rows
2026-09-30 00:19:39,292    9n3s-kdb3: fetched 3500 / 18330 rows
2026-09-30 00:19:41,163    9n3s-kdb3: fetched 4000 / 18330 rows
2026-09-30 00:19:41,885    9n3s-kdb3: fetched 4500 / 18330 rows
2026-09-30 00:19:42,486    9n3s-kdb3: fetched 5000 / 18330 rows
2026-09-30 00:19:44,285    9n3s-kdb3: fetched 5500 / 18330 rows
2026-09-30 00:19:44,889    9n3s-kdb3: fetched 6000 / 18330 rows
2026-09-30 00:19:45,437    9n3s-kdb3: fetched 6500 / 18330 rows
2026-09-30 00:19:45,962    9n3s-kdb3: fetched 7000 / 18330 rows
2026-09-30 00:19:48,276    9n3s-kdb3: fetch


==== COPY EVERYTHING BELOW AND PASTE IT TO CLAUDE ====
{
  "hospitals_measured": 2833,
  "hospitals_worse_than_expected": 2358,
  "pct_worse": 83.2,
  "excess_readmissions": 12245,
  "est_excess_cost": 186131432.0,
  "top_condition": [
    "Heart failure",
    39.7
  ],
  "top_state": [
    "FL",
    21607586.0
  ],
  "savings_if_half_gap_closed": 93065716.0,
  "period": [
    "07/01/2021",
    "06/30/2024"
  ],
  "conditions": [
    [
      "Heart failure",
      73813393.0,
      48.9
    ],
    [
      "Pneumonia",
      68787115.0,
      46.8
    ],
    [
      "Heart attack",
      17182920.0,
      49.7
    ],
    [
      "COPD",
      16290234.0,
      47.2
    ],
    [
      "Bypass surgery (CABG)",
      5182017.0,
      49.9
    ],
    [
      "Hip/knee replacement",
      4875753.0,
      47.9
    ]
  ],
  "top_states": [
    [
      "FL",
      21607586.0,
      93.9
    ],
    [
      "CA",
      19434851.0,
      89.3
    ],
    [
      "NY",
      13239558.0,
      85.5

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>